### Examen 20/02/2025: Evaluación 2 Examen 1: SVM y DecissionTree

<div style="border-style:groove;border-width:thin;padding:10px">
    <h3>NORMAS:</h3>
    <ul>
        <li>No se dispondrá de Internet durante el examen. Se podrá usar cualquier material descargado o en papel.</li>
        <li>Queda prohibida cualquier comunicación entre vosotros o con el exterior.</li>
        <li>Deberá grabarse toda la sesión desde antes de recibir el examen hasta después de entregarlo utilizando OBS de forma local. La grabación se entregará a la vez que el examen. Quedan prohibidos los atajos de teclado del OBS</li> 
        <li>No se puede utilizar ninguna herramienta que asista a la programación. Ni basada en IA ni de otro tipo.</li>
        <li>La entrega del examen se hará supervisada por el profesor y se anotará la hora exacta de entrega.</li>
        <li>Los móviles permanecerán durante toda la sesión sobre los PCs colocados con la pantalla hacia abajo.</li>
    </ul>
</div> 

<div style="border-style:groove;border-width:thin;padding:10px">
    <h3>EJERCICIO 1:</h3>
    <p>Importa el dataset 'stickers.csv'. Este dataset contiene datos de venta de pegatinas en varias tiendas de unos cuantos países. Se trata de predecir el número de elementos vendidos en cada caso. Debes conseguir los siguientes objetivos:</p>
    <ol>
        <li>Separa la fecha en dos columnas categóricas, una con el año y otra con el mes. Elimina la columna original con la fecha.</li>
        <li>Prepara los datos para que puedan ser usados para entrenar una IA.</li>
        <li>Entrena una Support Vector Machine y muestra una métrica adecuada del resultado. Se tendrá en cuenta lo bien que el sistema sea capaz de predecir el resultado.</li>
        <li>Realiza lo mismo pero con DecisionTree. Se tendrá en cuenta lo bien que el sistema sea capaz de predecir el resultado.</li>
        <li>Usa los modelos entrenados de ambos algoritmos para predecir el numero de ventas que se generarían con los siguientes datos de entrada:
        <table border>
                <tr>
                        <th>id</th>
                        <th>date</th>
                        <th>country</th>
                        <th>city</th>
                        <th>store</th>
                        <th>product</th>
                </tr>
                <tr>
                        <td>10</td>
                        <td>2012-08-03</td>
                        <td>Italy</td>
                        <td>Milano</td>
                        <td>Stickers for Less</td>
                        <td>Kerneler</td>
                </tr>
        </table>
        </li>
    </ol>
</div>

In [19]:
import pandas as pd
pegatinas = pd.read_csv("stickers.csv", sep=",")
#stickers["date"] = pd.to_datetime(stickers["date"], yearfirst=True)
#stickers["año"] = stickers["date"].dt.year
#stickers["mes"] = stickers["date"].dt.month

years = []
months = []
for i,row in pegatinas.iterrows():
    years.append(row['date'].split('-')[0])
    months.append(row['date'].split('-')[1])
pegatinas['year'] = years
pegatinas['month'] = months

In [20]:
pegatinas.drop(columns=["id", "date", "city"], inplace=True)
pegatinas.dropna(inplace=True)

stickers_dummies = pd.get_dummies(pegatinas, dtype=int)
stickers_dummies.dropna()

,num_sold,country_Canada,country_Finland,country_Italy,country_Kenya,country_Norway,country_Singapore,store_Discount Stickers,store_Premium Sticker Mart,store_Stickers for Less,...,year_2010,month_01,month_02,month_03,month_04,month_05,month_06,month_07,month_08,month_09
1,973.0,1,0,0,0,0,0,1,0,0,...,1,1,0,0,0,0,0,0,0,0
2,906.0,1,0,0,0,0,0,1,0,0,...,1,1,0,0,0,0,0,0,0,0
3,423.0,1,0,0,0,0,0,1,0,0,...,1,1,0,0,0,0,0,0,0,0
4,491.0,1,0,0,0,0,0,1,0,0,...,1,1,0,0,0,0,0,0,0,0
5,300.0,1,0,0,0,0,0,0,0,1,...,1,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22107,24.0,0,0,0,1,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
22108,17.0,0,0,0,1,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
22109,14.0,0,0,0,1,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
22110,168.0,0,0,0,0,1,0,1,0,0,...,1,0,0,0,0,0,0,0,0,1


In [21]:
stickers_dummies.corr(numeric_only=True)["num_sold"].abs().sort_values(ascending=False)

num_sold                      1.000000
country_Norway                0.460731
country_Kenya                 0.446813
product_Holographic Goose     0.352147
store_Discount Stickers       0.345647
product_Kaggle                0.325052
store_Premium Sticker Mart    0.246098
product_Kaggle Tiers          0.215778
product_Kerneler              0.179344
country_Italy                 0.120508
store_Stickers for Less       0.096918
country_Canada                0.066619
product_Kerneler Dark Mode    0.050909
month_01                      0.016416
month_09                      0.015099
month_05                      0.014869
month_08                      0.011140
month_07                      0.009634
month_04                      0.008771
country_Singapore             0.006550
month_02                      0.006018
month_03                      0.004667
month_06                      0.004212
country_Finland               0.000997
year_2010                          NaN
Name: num_sold, dtype: fl

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
escalado = StandardScaler()

X = stickers_dummies.drop(columns="num_sold", axis=1)
y = stickers_dummies["num_sold"]

X_scaled = escalado.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [23]:
# Entrenar el modelo de SVR con Kernel RBF
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

svm_model = SVR(kernel='rbf', C=60)
#svr_lnr = SVR(kernel="linear", C=60)
#svr_poly = SVR(kernel="poly", degree=1, C=60)

svm_model.fit(X_train, y_train)
#svr_lnr.fit(X_train,y_train)
#svr_poly.fit(X_train,y_train)

y_pred_rbf = svm_model.predict(X_test)
#y_pred_lnr = svr_lnr.predict(X_test)
#y_pred_poly = svr_poly.predict(X_test)

# Evaluar el modelo SVR
from sklearn.metrics import r2_score
print(r2_score(y_test, y_pred_rbf))
#print(r2_score(y_test, y_pred_lnr))
#print(r2_score(y_test, y_pred_poly))

0.9686278308359584


In [24]:
from sklearn.tree import DecisionTreeRegressor

# Entrenar el modelo con DecisionTree
treeclf = DecisionTreeRegressor(max_depth=100, random_state=42)
treeclf.fit(X_train, y_train)

y_pred_tree = treeclf.predict(X_test)

# Evaluar el modelo
from sklearn.metrics import r2_score
print(r2_score(y_test, y_pred_tree))

0.9742537131518443


<div style="border-style:groove;border-width:thin;padding:10px">
    <h3>EJERCICIO 2:</h3>
    <p>Importa los datasets 'setas1.csv' y 'setas2.csv'. Éstos contienen datos sobre setas que permiten determinar si son venenosas o no (columna class). Debes conseguir los siguientes objetivos:</p>
    <ol>
        <li>Junta los dos datasets en uno solo.</li>
        <li>Modifica la columna class para que tenga los valores 0 (e) y 1(p).</li>
        <li>Prepara los datos para que puedan ser usados para entrenar una IA.</li>
        <li>Entrena una Support Vector Machine y muestra una métrica adecuada del resultado. Se tendrá en cuenta lo bien que el sistema sea capaz de predecir el resultado.</li>
        <li>Realiza lo mismo pero con DecisionTree. Se tendrá en cuenta lo bien que el sistema sea capaz de predecir el resultado.</li>
        <li>Usa los modelos entrenados de ambos algoritmos para predecir si una seta con los siguientes datos de entrada sería venenosa:
        <table border>
                <tr>
                        <th>id</th>
                        <th>cap-color</th>
                        <th>does-bruise-or-bleed</th>
                        <th>stem-height</th>
                        <th>reg</th>
                        <th>has-ring</th>
                        <th>habitat</th>
                        <th>season</th>
                </tr>
                <tr>
                        <td>37</td>
                        <td>n</td>
                        <td>t</td>
                        <td>5.8</td>
                        <td>37</td>
                        <td>f</td>
                        <td>l</td>
                        <td>a</td>
                </tr>
        </table>
        </li>
    </ol>
</div>

In [25]:
import pandas as pd

setas1 = pd.read_csv('setas1.csv', sep=',')
setas2 = pd.read_csv('setas2.csv', sep=',')

df_setas = pd.merge(left=setas1,right=setas2,how='inner',left_on='id',right_on='reg')
df_setas.isna().sum()
df_setas

,id,class,cap-color,does-bruise-or-bleed,stem-height,reg,has-ring,habitat,season
0,0,e,u,f,4.51,0,f,d,a
1,1,p,o,f,4.79,1,t,d,w
2,2,e,b,f,6.85,2,f,l,w
3,3,e,g,f,4.16,3,f,d,u
4,4,e,w,f,3.37,4,f,g,a
...,...,...,...,...,...,...,...,...,...
19994,19994,e,n,t,8.00,19994,t,d,u
19995,19995,p,n,f,4.64,19995,f,d,s
19996,19996,e,n,t,9.05,19996,t,d,a
19997,19997,p,w,f,3.50,19997,f,g,a


In [26]:
df_setas["does-bruise-or-bleed"] = df_setas["does-bruise-or-bleed"].replace({"t": True, "f": False})
df_setas["has-ring"] = df_setas["has-ring"].replace({"t": True, "f": False})
df_setas.drop(columns=["id", "reg"], inplace=True, axis=1)
df_setas

/tmp/ipykernel_22553/3555363565.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_setas["does-bruise-or-bleed"] = df_setas["does-bruise-or-bleed"].replace({"t": True, "f": False})


,class,cap-color,does-bruise-or-bleed,stem-height,has-ring,habitat,season
0,e,u,False,4.51,False,d,a
1,p,o,False,4.79,True,d,w
2,e,b,False,6.85,False,l,w
3,e,g,False,4.16,False,d,u
4,e,w,False,3.37,False,g,a
...,...,...,...,...,...,...,...
19994,e,n,True,8.00,True,d,u
19995,p,n,False,4.64,False,d,s
19996,e,n,True,9.05,True,d,a
19997,p,w,False,3.50,False,g,a


In [27]:
setas_dummies = pd.get_dummies(df_setas,columns=['cap-color',
       'does-bruise-or-bleed', 'has-ring', 'habitat',
       'season'],dtype='int')

setas_dummies.replace({'e':0,'p':1},inplace=True)
setas_dummies.head()

/tmp/ipykernel_22553/2043515732.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  setas_dummies.replace({'e':0,'p':1},inplace=True)


,class,stem-height,cap-color_b,cap-color_e,cap-color_g,cap-color_i,cap-color_k,cap-color_l,cap-color_n,cap-color_o,...,habitat_n,habitat_p,habitat_s,habitat_u,habitat_w,habitat_y,season_a,season_s,season_u,season_w
0,0,4.51,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,1,4.79,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,1
2,0,6.85,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,4.16,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,3.37,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [28]:
setas_dummies.corr()['class'].abs().sort_values(ascending=False)[1:20]

season_w                      0.125613
cap-color_e                   0.125170
cap-color_b                   0.115571
cap-color_r                   0.108967
cap-color_n                   0.104597
habitat_g                     0.098731
habitat_w                     0.089566
cap-color_o                   0.085767
habitat_l                     0.077584
season_s                      0.077526
habitat_p                     0.067059
cap-color_g                   0.059539
cap-color_y                   0.059337
season_u                      0.057364
has-ring_True                 0.054367
has-ring_False                0.054235
stem-height                   0.051600
cap-color_p                   0.051518
does-bruise-or-bleed_False    0.050573
Name: class, dtype: float64

In [29]:
# Asignar valor a X, Y (escalando X)
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
X = escalador.fit_transform(setas_dummies.drop(columns=["class"]))
y = setas_dummies["class"]

# Dividir en train y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
# Entrenando con Random Forest
from sklearn.ensemble import RandomForestClassifier
tree_clf = RandomForestClassifier(random_state=42, n_estimators=1000, n_jobs=-1, bootstrap=True)
tree_clf.fit(X_train, y_train)

# Evaluar el modelo
from sklearn.metrics import classification_report, confusion_matrix
y_pred = tree_clf.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Accuracy:", tree_clf.score(X_test, y_test))

              precision    recall  f1-score   support

           0       0.76      0.73      0.75      1842
           1       0.78      0.80      0.79      2158

    accuracy                           0.77      4000
   macro avg       0.77      0.77      0.77      4000
weighted avg       0.77      0.77      0.77      4000

[[1352  490]
 [ 434 1724]]
Accuracy: 0.769


In [31]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
svc_rbf = SVC(kernel="rbf", C=2000, coef0=1)

svc_rbf.fit(X_train,y_train)

y_pred_rbf = svc_rbf.predict(X_test)

print(accuracy_score(y_test, y_pred_rbf))

0.795


In [32]:
setas1 = pd.read_csv('setas1.csv', sep=',')
setas2 = pd.read_csv('setas2.csv', sep=',')

df_setas = pd.merge(left=setas1,right=setas2,how='inner',left_on='id',right_on='reg')
df_setas.isna().sum()
df_setas.drop(['id','reg'],axis=1,inplace=True)

setas_nuevas = pd.get_dummies(df_setas, dtype='int')
setas_nuevas.replace({'e':0,'p':1},inplace=True)
print(X_test[0])

[-1.24909232 -0.13710492 -0.25864617  3.62955035 -0.00707142 -0.14213019
 -0.11228355 -0.88088161 -0.24540521 -0.17692249 -0.15597766 -0.1563137
 -0.37164157 -0.37442468  0.45706686 -0.45706686  0.56551274 -0.56543573
 -0.00707142 -0.00707142 -1.52022785  2.44003181 -0.20252942 -0.24160327
 -0.2279691  -0.00707142 -0.07368576 -0.00707142 -0.04476727 -0.08151186
 -0.00707142  1.01597843 -0.21971686 -0.76811354 -0.3160277 ]


Criterio de calificación: 

Cada ejercicio y opción debidamente desarrollada se califica con los puntos especificados en la siguiente tabla. La suma de los puntos calificados para cada opción será la calificación de la prueba:


| Apartado | Calificación máxima | Calificación obtenida |
| :-- | --- | --- |
| Ej. 1.1 Separa fecha. | 0,25 |  |
| Ej. 1.2 Prepara datos. | 2 |  |
| Ej. 1.3 SVM | 1 |  |
| Ej. 1.4 DecissionTree | 1 |  |
| Ej. 1.5 Entrada nueva | 0,5 |  |
| Ej. 2.1 Unir Datasets | 0,5 |  |
| Ej. 2.2 Modifica columna class | 0,25 |  |
| Ej. 2.3 Prepara datos | 2 |  |
| Ej. 2.4 SVM | 1 |  |
| Ej. 2.5 DecissionTree | 1 |  |
| Ej. 2.6 Entrada nueva | 0,5 |  |

